# Solutions to Exercises on Conditional Independence

**1.** Consider the following situations. For each pair of quantities decide whether they are

* *independent*;
* *dependent*, but independent when conditioned on a third quantity (and if so, what is the third quantity?); or
* *independent*, but dependent when conditioned on a third quantity.

Say which of the structures from the notes (direct link, chain, fork or collider) describes the situation. Where there is a confounder, name it.

a) Height and shoe size of children.  
b) Reading ability and eye colour.  
c) Journey times in Belfast and weather conditions.  
d) Centimetres of rainfall and hours of darkness (over a year in one location).  
e) Student grades and annual rainfall in their home town (assume the towns are otherwise identical).  
f) Crop yield and sales of sunblock.  
g) Studying mathematics and careers in finance.  
h) Studying geography and careers in finance.  
i) Burglary and earthquake, given that a house alarm is ringing.  
j) Ice-cream sales and the number of cases of sunburn.  
k) Smoking and lung cancer, given the amount of tar in the lungs.  
l) The results of two fair coin tosses, given that at least one landed heads.

**Solution**

| | Relationship | Third quantity | Structure |
|:--|:--|:--|:--|
| a) Height and shoe size | Dependent; conditionally independent given the third quantity | Overall body size (the confounder) | Fork |
| b) Reading ability and eye colour | Independent | | None |
| c) Journey times and weather | Dependent | | Direct link: bad weather makes journeys longer |
| d) Rainfall and hours of darkness | Dependent; conditionally independent given the third quantity | Time of year (the confounder) | Fork |
| e) Grades and annual rainfall | Independent | | None |
| f) Crop yield and sunblock sales | Dependent; conditionally independent given the third quantity | The weather, in particular sunshine (the confounder) | Fork |
| g) Studying mathematics and careers in finance | Dependent | | Direct link |
| h) Studying geography and careers in finance | Independent (or very nearly so) | | None |
| i) Burglary and earthquake, given the alarm | Independent, but dependent given the third quantity | The alarm (the common effect) | Collider |
| j) Ice-cream sales and sunburn | Dependent; conditionally independent given the third quantity | Hours of sunshine (the confounder) | Fork |
| k) Smoking and lung cancer, given tar | Dependent; conditionally independent given the third quantity | Tar in the lungs (a mediator) | Chain |
| l) Two coin tosses, given at least one head | Independent, but dependent given the third quantity | "At least one head" (the common effect) | Collider |

Notes. In (a) it is the overall body size, of which height and shoe size are both consequences, that is the common cause; age alone would not remove the dependence, since children of the same age differ in size. Quantities such as body size are not usually observed directly, but the conditional independence holds in principle. In (c) and (g) there is a direct link and so no third quantity makes the two independent. (i) and (l) are examples of "explaining away", which we examine numerically in exercises 3 and 4.

**2.** The joint distribution `PXYZ[x][y][z]` of three binary variables $X$, $Y$ and $Z$ is given below. The functions `independent` and `cond_independent` (also given) test whether two variables are independent, or conditionally independent given a third.

a) Compute the marginal distributions $P(Z)$, $P(X,Y)$, $P(X)$ and $P(Y)$. Are $X$ and $Y$ independent? Do not simply call `independent`: compute $P(X)P(Y)$ and compare it with $P(X,Y)$.

b) Compute $P(X,Y\vert Z)$ for each value of $Z$. Are $X$ and $Y$ conditionally independent given $Z$? Check your answer by comparing $P(X,Y\vert Z)$ with $P(X\vert Z)P(Y\vert Z)$.

c) Verify that $P(X\vert Y,Z)=P(X\vert Z)$.

d) Which of the structures chain, fork or collider could have generated this distribution?

In [1]:
import numpy as np

def independent(PXY, tol=1e-9):
    """Test whether X and Y are independent, for a joint distribution PXY[x][y]."""
    return np.allclose(PXY, np.outer(PXY.sum(axis=1), PXY.sum(axis=0)), atol=tol)

def cond_independent(PXYZ, tol=1e-9):
    """Test whether X and Y are conditionally independent given Z, for a joint distribution PXYZ[x][y][z]."""
    PZ = PXYZ.sum(axis=(0, 1))
    PXY_Z = PXYZ / PZ[None, None, :]                  # P(X, Y | Z)
    PX_Z = PXY_Z.sum(axis=1)                          # P(X | Z)[x][z]
    PY_Z = PXY_Z.sum(axis=0)                          # P(Y | Z)[y][z]
    return np.allclose(PXY_Z, np.einsum('xz,yz->xyz', PX_Z, PY_Z), atol=tol)

PXYZ = np.array([[[0.432, 0.048],
                  [0.048, 0.072]],
                 [[0.108, 0.112],
                  [0.012, 0.168]]])
print(PXYZ.sum())

1.0


a) Marginal distributions

In [2]:
PZ = PXYZ.sum(axis=(0, 1))
PXY = PXYZ.sum(axis=2)
PX = PXY.sum(axis=1)
PY = PXY.sum(axis=0)
print("P(Z) =", PZ)
print("P(X,Y):")
print(PXY)
print("P(X) P(Y):")
print(np.round(np.outer(PX, PY), 4))
print("X and Y independent?", np.allclose(PXY, np.outer(PX, PY)))

P(Z) = [0.6 0.4]
P(X,Y):
[[0.48 0.12]
 [0.22 0.18]]
P(X) P(Y):
[[0.42 0.18]
 [0.28 0.12]]
X and Y independent? False


$P(X,Y)\neq P(X)P(Y)$ (for example $P(X=1,Y=1)=0.18$ but $P(X=1)P(Y=1)=0.4\times0.3=0.12$), so $X$ and $Y$ are **not** independent.

b) Conditional distributions

In [3]:
PXY_Z = PXYZ / PZ[None, None, :]             # P(X,Y | Z)[x][y][z]
PX_Z = PXY_Z.sum(axis=1)                     # P(X | Z)[x][z]
PY_Z = PXY_Z.sum(axis=0)                     # P(Y | Z)[y][z]
for z in (0, 1):
    print(f"P(X,Y | Z={z}):")
    print(np.round(PXY_Z[:, :, z], 4))
    print(f"P(X | Z={z}) P(Y | Z={z}):")
    print(np.round(np.outer(PX_Z[:, z], PY_Z[:, z]), 4))
print("X and Y conditionally independent given Z?", np.allclose(PXY_Z, np.einsum('xz,yz->xyz', PX_Z, PY_Z)))
print("Check with function:", cond_independent(PXYZ))

P(X,Y | Z=0):
[[0.72 0.08]
 [0.18 0.02]]
P(X | Z=0) P(Y | Z=0):
[[0.72 0.08]
 [0.18 0.02]]
P(X,Y | Z=1):
[[0.12 0.18]
 [0.28 0.42]]
P(X | Z=1) P(Y | Z=1):
[[0.12 0.18]
 [0.28 0.42]]
X and Y conditionally independent given Z? True
Check with function: True


For each value of $Z$, $P(X,Y\vert Z)$ equals $P(X\vert Z)P(Y\vert Z)$, so $X\perp\!\!\!\perp Y\mid Z$.

c) Verify $P(X\vert Y,Z)=P(X\vert Z)$

In [4]:
PYZ = PXYZ.sum(axis=0)                        # P(Y, Z)[y][z]
PX_YZ = PXYZ / PYZ[None, :, :]               # P(X | Y, Z)[x][y][z]
for y in (0, 1):
    for z in (0, 1):
        print(f"P(X=1 | Y={y}, Z={z}) = {PX_YZ[1, y, z]:.4f}   P(X=1 | Z={z}) = {PX_Z[1, z]:.4f}")

P(X=1 | Y=0, Z=0) = 0.2000   P(X=1 | Z=0) = 0.2000
P(X=1 | Y=0, Z=1) = 0.7000   P(X=1 | Z=1) = 0.7000
P(X=1 | Y=1, Z=0) = 0.2000   P(X=1 | Z=0) = 0.2000
P(X=1 | Y=1, Z=1) = 0.7000   P(X=1 | Z=1) = 0.7000


d) The dependence of $X$ and $Y$ disappears when conditioning on $Z$, so the distribution is either a *chain* ($X\to Z\to Y$ or $Y\to Z\to X$) or a *fork* ($X\leftarrow Z\to Y$). A collider would have $X$ and $Y$ independent. In fact this distribution was generated as a fork, $P(Z)P(X\vert Z)P(Y\vert Z)$, with $P(Z)=(0.6,0.4)$. The data alone cannot distinguish between a chain and a fork, since both imply the same independence relations.

**3.** Two fair coins are tossed. Let $X$ and $Y$ be the results (0 = tails, 1 = heads) and let $Z=1$ if *at least one* coin landed heads, and $Z=0$ otherwise.

a) Write down the joint distribution `PXYZ[x][y][z]`.

b) Show that $X$ and $Y$ are independent.

c) Compute $P(X,Y\vert Z=1)$. Are $X$ and $Y$ independent given $Z=1$? Compare $P(X=1,Y=1\vert Z=1)$ with $P(X=1\vert Z=1)P(Y=1\vert Z=1)$.

d) Compute $P(X=1\vert Y=0,Z=1)$ and compare it with $P(X=1\vert Z=1)$. Explain in words why learning the result of the second coin changes our belief about the first.

a) and b)

In [5]:
PXYZ = np.zeros((2, 2, 2))
for x in (0, 1):
    for y in (0, 1):
        z = int(x == 1 or y == 1)                 # Z = 1 if at least one head
        PXYZ[x, y, z] = 0.25
print("P(X,Y):")
PXY = PXYZ.sum(axis=2)
print(PXY)
print("X and Y independent?", independent(PXY))

P(X,Y):
[[0.25 0.25]
 [0.25 0.25]]
X and Y independent? True


c)

In [6]:
PZ = PXYZ.sum(axis=(0, 1))
PXY_Z1 = PXYZ[:, :, 1] / PZ[1]               # P(X, Y | Z=1)
PX_Z1 = PXY_Z1.sum(axis=1)
PY_Z1 = PXY_Z1.sum(axis=0)
print("P(X,Y | Z=1):")
print(np.round(PXY_Z1, 4))
print("P(X | Z=1) P(Y | Z=1):")
print(np.round(np.outer(PX_Z1, PY_Z1), 4))
print("P(X=1,Y=1 | Z=1) =", round(PXY_Z1[1, 1], 4), "  P(X=1|Z=1) P(Y=1|Z=1) =", round(PX_Z1[1] * PY_Z1[1], 4))
print("X and Y independent given Z=1?", independent(PXY_Z1))

P(X,Y | Z=1):
[[0.     0.3333]
 [0.3333 0.3333]]
P(X | Z=1) P(Y | Z=1):
[[0.1111 0.2222]
 [0.2222 0.4444]]
P(X=1,Y=1 | Z=1) = 0.3333   P(X=1|Z=1) P(Y=1|Z=1) = 0.4444
X and Y independent given Z=1? False


d)

In [7]:
PX_Y0Z1 = PXYZ[:, 0, 1] / PXYZ[:, 0, 1].sum()   # P(X | Y=0, Z=1)
print("P(X=1 | Y=0, Z=1) =", PX_Y0Z1[1])
print("P(X=1 | Z=1)      =", round(PX_Z1[1], 4))

P(X=1 | Y=0, Z=1) = 1.0
P(X=1 | Z=1)      = 0.6667


The coins are independent, but if we know that at least one landed heads and that the second coin landed tails, then the first coin must have landed heads: $P(X=1\vert Y=0,Z=1)=1$, compared with $P(X=1\vert Z=1)=2/3$. Knowing the result of one coin tells us something about the other because both are possible causes of the same observed effect. This is a collider, $X\to Z\leftarrow Y$.

**4.** *(Explaining away)* The grass is wet ($W$) if it has rained ($R$) or the sprinkler was on ($S$). Rain and the sprinkler are independent, with $P(R)=0.2$ and $P(S)=0.3$. The probability that the grass is wet is

| | $P(W=1\vert R,S)$ |
|:--|:-:|
| $\lnot R,\lnot S$ | 0.05 |
| $R,\lnot S$ | 0.90 |
| $\lnot R,S$ | 0.80 |
| $R,S$ | 0.99 |

a) Write down the factorisation of $P(R,S,W)$ and identify the structure.

b) Compute $P(R)$, $P(R\vert W=1)$, $P(R\vert W=1,S=1)$ and $P(R\vert W=1,S=0)$.

c) Explain why $P(R\vert W=1,S=1)$ is smaller than $P(R\vert W=1)$, and why $P(R\vert W=1,S=0)$ is larger.

a) $P(R,S,W)=P(R)P(S)P(W\vert R,S)$. This is a collider, with rain and the sprinkler as independent causes of wet grass.

b)

In [8]:
PR = np.array([0.8, 0.2])
PS = np.array([0.7, 0.3])

# PW_RS[w][r][s]; P(W=1 | r, s) is given for (r,s) = (0,0), (0,1), (1,0), (1,1)
PW1_RS = np.array([[0.05, 0.80],
                   [0.90, 0.99]])
PW_RS = np.array([1 - PW1_RS, PW1_RS])

PRSW = np.einsum('r,s,wrs->rsw', PR, PS, PW_RS)       # joint, PRSW[r][s][w]
PW = PRSW.sum(axis=(0, 1))
PRS_W = PRSW[:, :, 1] / PW[1]                         # P(R, S | W=1)[r][s]

print(f"P(R=1)               = {PR[1]:.4f}")
print(f"P(R=1 | W=1)         = {PRS_W.sum(axis=1)[1]:.4f}")
print(f"P(R=1 | W=1, S=1)    = {PRS_W[1, 1] / PRS_W[:, 1].sum():.4f}")
print(f"P(R=1 | W=1, S=0)    = {PRS_W[1, 0] / PRS_W[:, 0].sum():.4f}")

P(R=1)               = 0.2000
P(R=1 | W=1)         = 0.4573
P(R=1 | W=1, S=1)    = 0.2363
P(R=1 | W=1, S=0)    = 0.8182


c) Wet grass raises the probability of rain from 20% to 46%. If we then learn that the sprinkler was on, the sprinkler provides an alternative explanation for the wet grass, so rain becomes less likely (24%). If instead we learn that the sprinkler was off, rain is the only plausible explanation and becomes much more likely (82%). Rain and the sprinkler are independent, but they are dependent given the wet grass.

**5.** *(Counting parameters)*

a) A binary class variable $C$ has 20 binary features $F_1,\dots,F_{20}$ that are conditionally independent given $C$. How many free parameters are needed for the full joint distribution $P(C,F_1,\dots,F_{20})$, and how many for the factorised form $P(C)\prod_iP(F_i\vert C)$?

b) For three binary variables, how many free parameters are needed for the chain $P(X)P(Z\vert X)P(Y\vert Z)$, the fork $P(Z)P(X\vert Z)P(Y\vert Z)$ and the collider $P(X)P(Y)P(Z\vert X,Y)$? Compare with the 7 needed for the full joint distribution.

In [9]:
# a) Full joint of 21 binary variables vs naive Bayes factorisation
print("Full joint:  ", f"{2**21 - 1:,}")
print("Factorised:  ", 1 + 2*20)

# b) Three binary variables
print("Chain    P(X)P(Z|X)P(Y|Z):   ", 1 + 2 + 2)
print("Fork     P(Z)P(X|Z)P(Y|Z):   ", 1 + 2 + 2)
print("Collider P(X)P(Y)P(Z|X,Y):   ", 1 + 1 + 4)
print("Full joint:                  ", 2**3 - 1)

Full joint:   2,097,151
Factorised:   41
Chain    P(X)P(Z|X)P(Y|Z):    5
Fork     P(Z)P(X|Z)P(Y|Z):    5
Collider P(X)P(Y)P(Z|X,Y):    6
Full joint:                   7


a) The full joint distribution needs $2^{21}-1=2{,}097{,}151$ parameters and the factorised form needs $1+2\times20=41$.

b) The chain and the fork each need 5 parameters (1 for the root, and 2 for each of the two conditional distributions $P(\cdot\vert\text{one binary parent})$) and the collider needs 6 (1 + 1 + 4 for $P(Z\vert X,Y)$), all fewer than the 7 for the full joint distribution. The saving is small here but grows exponentially with the number of variables.